# 13 DAW Playground

One notebook cell builds a small in-Jupyter DAW: global transport, multitrack timeline, sequencers, sampler pads, keyboard input, mixer channel strips, built-in insert effects, and browser recording.


In [ ]:
import math

import ipywidgets as widgets
from IPython.display import display

from nbplay import (
    EffectPlugin,
    KeyboardWidget,
    PadWidget,
    SamplerWidget,
    SequencerWidget,
    Session,
    SettingsWidget,
    SynthWidget,
)


def decaying_tone(freq=440, seconds=0.22, sample_rate=44100, drive=0.9):
    frames = int(sample_rate * seconds)
    return [
        math.sin(2 * math.pi * freq * i / sample_rate) * drive * (1 - i / frames) ** 2
        for i in range(frames)
    ]

settings = SettingsWidget(sample_rate=44100, channels=2, buffer_size=512)
session = Session(bpm=126.0, time_signature=(4, 4))
session.timeline.length = 64
session.timeline.count_in_bars = 1
session.timeline.auto_extend_recording = True
session.timeline.recording_extend_bars = 16

lead = SynthWidget(oscillator_type="saw", frequency=440.0, amplitude=0.42)
bass = SynthWidget(oscillator_type="square", frequency=110.0, amplitude=0.46)
drums = SamplerWidget(attack=0.002, decay=0.06, sustain=0.2, release=0.08, pad_count=8, max_voices=8)
drums.load_sample(decaying_tone(96, seconds=0.28, drive=1.0), sample_rate=44100, root_note=36, name="Kick-ish")

lead_seq = SequencerWidget(length=16, num_voices=2, bpm=session.transport.bpm)
for i, note in enumerate([72, 76, 79, 84, 83, 79, 76, 72, 71, 74, 78, 83, 81, 78, 74, 71]):
    lead_seq.set_step(i, note=note, velocity=88 + (i % 4) * 6, active=i % 2 == 0, voice=0)
for i, note in enumerate([60, 60, 64, 64, 67, 67, 64, 64, 59, 59, 62, 62, 66, 66, 62, 62]):
    lead_seq.set_step(i, note=note, velocity=58, active=i % 4 == 0, voice=1)

bass_seq = SequencerWidget(length=16, bpm=session.transport.bpm)
for i, note in enumerate([36, 36, 43, 36, 34, 34, 41, 34, 31, 31, 38, 31, 34, 34, 41, 34]):
    bass_seq.set_step(i, note=note, velocity=104 if i % 4 == 0 else 72, active=i % 2 == 0)

drum_seq = SequencerWidget(length=16, bpm=session.transport.bpm)
for i in [0, 4, 8, 12]:
    drum_seq.set_step(i, note=36, velocity=118, active=True)
for i in [3, 7, 11, 15]:
    drum_seq.set_step(i, note=38, velocity=72, active=True)

lead_track = session.add_track("Lead", lead_seq, lead)
bass_track = session.add_track("Bass", bass_seq, bass)
drum_track = session.add_track("Drums", drum_seq, drums)

session.mixer.set_channel_gain(lead_track.mixer_channel, 0.74)
session.mixer.set_channel_pan(lead_track.mixer_channel, 0.18)
session.mixer.add_channel_effect(lead_track.mixer_channel, EffectPlugin("filter", filter_type="lowpass", frequency=3600, q=0.7))
session.mixer.add_channel_effect(lead_track.mixer_channel, EffectPlugin("reverb", seconds=1.4, decay=2.6, wet=0.16))

session.mixer.set_channel_gain(bass_track.mixer_channel, 0.88)
session.mixer.set_channel_pan(bass_track.mixer_channel, -0.12)
session.mixer.add_channel_effect(bass_track.mixer_channel, EffectPlugin("compressor", threshold=-24, ratio=5, release=0.2))

session.mixer.set_channel_gain(drum_track.mixer_channel, 0.68)
session.mixer.add_channel_effect(drum_track.mixer_channel, EffectPlugin("compressor", threshold=-18, ratio=6, attack=0.003, release=0.12))
session.mixer.add_channel_effect(drum_track.mixer_channel, EffectPlugin("delay", time=0.125, feedback=0.18, wet=0.12))
session.mixer.master_gain = 0.82
session.mixer.add_master_effect(EffectPlugin("limiter", threshold=-1, release=0.04))

for track in (lead_track, bass_track, drum_track):
    session.timeline.add_clip(f"{track.name} pattern", track_index=track.mixer_channel, start=0, duration=16, loop=True, source="sequencer")
session.timeline.arm_track(drum_track.mixer_channel, True, exclusive=True)
timeline_tracks = [dict(track) for track in session.timeline.tracks]
timeline_tracks[drum_track.mixer_channel]["monitor"] = True
session.timeline.tracks = timeline_tracks

keyboard = KeyboardWidget(upper_octave=4, lower_octave=3, velocity=96)
keyboard.connect_sequencer(lead_seq)
keyboard.connect_sampler(drums)

pads = PadWidget(rows=2, cols=4, velocity=116)
pads.connect_sampler(drums)

transport_tab = widgets.VBox([settings, session.transport, session.timeline])
source_tab = widgets.VBox([lead, bass, drums])
sequence_tab = widgets.VBox([lead_seq, bass_seq, drum_seq])
input_tab = widgets.VBox([keyboard, pads])

tabs = widgets.Tab([transport_tab, source_tab, sequence_tab, input_tab, session.mixer])
for index, title in enumerate(["Transport", "Sources", "Sequencers", "Inputs", "Mixer"]):
    tabs.set_title(index, title)

display(tabs)
print("Record target:", session.timeline.tracks[drum_track.mixer_channel]["name"])
print(session)


Use the transport tab to start/stop the session. Use the timeline record button to capture microphone audio on the armed track. Recorded clips play back through the session mixer while the notebook view is alive.
